# 00 · Kernel check

This notebook establishes nothing about Fieldwork. It answers one question: **does this kernel actually provide the packages the checks need, and which versions?**

A package index is not a demonstrated run. The versions printed below are the evidence; anything a document claims about availability is only a reason to try. Record this output alongside any result produced in this kernel.

In [ ]:
# pyproj, shapely, rasterio and friends ship in the Pyodide distribution, not as PyPI
# wheels on this site, so they load with pyodide_js.loadPackage. piplite.install would
# look in the site's local wheel index and fail. Outside Pyodide this cell does nothing.
NEEDED = ["pyproj", "shapely", "rasterio", "geopandas", "fiona", "numpy", "pandas", "h3", "scipy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    print("Falling back to piplite for anything still missing.")
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)


In [ ]:
import importlib, platform, sys

print(f"runtime   CPython {platform.python_version()} on {platform.system()} {platform.machine()}")
print(f"pyodide   {'yes' if 'pyodide' in sys.modules or platform.system() == 'Emscripten' else 'no'}")
print()
for name in ["pyproj", "shapely", "rasterio", "geopandas", "fiona", "numpy", "pandas", "h3", "scipy"]:
    try:
        module = importlib.import_module(name)
        print(f"{name:11s} {getattr(module, '__version__', 'unknown')}")
    except Exception as error:
        print(f"{name:11s} ABSENT ({type(error).__name__}: {error})")

In [ ]:
# PROJ and GEOS carry their own versions, which are what the comparisons actually exercise.
import pyproj
print("PROJ", pyproj.proj_version_str)
print("pyproj data dir", pyproj.datadir.get_data_dir())
try:
    import shapely
    print("GEOS", shapely.geos_version_string)
except Exception as error:
    print("GEOS unavailable:", error)

## What this does not show

An import is not a working transformation. `pyproj` without its PROJ data directory can import and then fail on a transform that needs a grid. The checks in this lab deliberately cover projection changes on a single datum, which need no grid; a datum shift would, and must error rather than silently download one.